In [2]:
#Assignment 6: Gabriel Maldonado

In [4]:
import pygame
import moderngl
import numpy
import math
import glm
from LoadObject import getObjectData

pygame.init()

width = 600
height = 600
pygame.display.gl_set_attribute(pygame.GL_CONTEXT_PROFILE_MASK, pygame.GL_CONTEXT_PROFILE_CORE)
display = pygame.display.set_mode((width, height), flags= pygame.OPENGL | pygame.DOUBLEBUF | pygame.RESIZABLE)
display = pygame.display.set_caption(title = "Assignment 6 - Gabriel Maldonado")
gl = moderngl.get_context()
gl.info["GL_VERSION"]


texture_position_data = [
    -1.0, -1.0,
    1.0, -1.0,
    -1.0, 1.0,
     1.0, -1.0,
    1.0, 1.0,
    -1.0, 1.0
]
texture_coords = [
    0.0, 0.0,
    1.0, 0.0,
    0.0, 1.0,
    1.0, 0.0,
    1.0, 1.0,
    0.0, 1.0
]

texture_geometry = numpy.array(texture_coords).astype("float32")
texture_positions = numpy.array(texture_position_data).astype("float32")

texture_geometry_buffer = gl.buffer(texture_geometry)
texture_positions_buffer = gl.buffer(texture_positions)
    

texture_vertex_shader_code = '''
    #version 330 core

    layout(location=0);
    in vec3 positions;
    in vec2 inUV;
    out vec2 uv;
    uniform mat2 aspectMatrix;
    uniform mat4 viewMatrix;
    uniform mat4 perspectiveMatrix;

    void main(){
        gl_Position = perspectiveMatrix * viewMatrix * vec4(positions,1);
        uv = inUV;
}

'''


texture_fragment_shader_code = '''
    #version 330 core

    in vec2 uv;
    uniform sampler2D tex;
    out vec4 tColor;

    void main(){
        tColor = texture(tex, uv);
}    
'''

ctx = moderngl.create_context()

textureProgram = gl.program(
    vertex_shader = texture_vertex_shader_code,
    fragment_shader = texture_fragment_shader_code
)

shape_data, bounds = getObjectData("teapot_with_texCoords-2.obj", texture=True)
vars(bounds)


shape_data_array = numpy.array(shape_data).astype("float32")
shape_data_buffer = gl.buffer(shape_data_array)


textureRenderable = gl.vertex_array(textureProgram, [
    (shape_data_buffer, "3f 2f", "positions", "inUV")]
)


texture_img = pygame.image.load("brick.jpg")
texture_data = pygame.image.tobytes(texture_img, "RGBA", True)
texture_width = texture_img.get_width()
texture_height = texture_img.get_height()
texture_dim = (texture_width, texture_height)

texture = ctx.texture(texture_dim, 4, texture_data)
texture.build_mipmaps
repeat_x = True
repeat_y = True
sampler = ctx.sampler(texture = texture, repeat_x=True, repeat_y=True, filter = (gl.LINEAR, gl.LINEAR))
sampler.texture = texture

camera_elevation_angle = math.radians(60)
camera_y_angle = 0
distance = 2 * bounds.radius


gl.enable(gl.DEPTH_TEST)
running = True
clock = pygame.time.Clock()
angle = 0

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT or (event.type == pygame.KEYDOWN and event.key == 27):
            running = False
        elif (event.type == pygame.WINDOWRESIZED):
            width = event.x
            height = event.y
    aspect = width/height

    eye_y = distance * math.cos(camera_elevation_angle)

    eye_radius = distance * math.sin(camera_elevation_angle)
    eye_x = eye_radius * math.cos(math.radians(camera_y_angle))
    eye_z = eye_radius * math.sin(math.radians(camera_y_angle))

    eye_point = glm.vec3(eye_x, eye_y, eye_z)

    target_point = glm.vec3(0, 0, 0)
    up_vector = glm.vec3(0, 1, 0)

    camera_trans_matrix = glm.lookAt(eye_point, target_point, up_vector)
    camera_perspective_matrix = glm.perspective(math.radians(45), aspect, bounds.radius, 3 * bounds.radius)         
            
    gl.clear(0,0,0)

    sampler.use(0)
    textureProgram["tex"] = 0
    textureProgram["viewMatrix"].write(camera_trans_matrix)
    textureProgram["perspectiveMatrix"].write(camera_perspective_matrix)
    textureRenderable.render()

    
    pygame.display.flip()
    clock.tick(30)
    camera_y_angle = camera_y_angle + 1
    if(camera_y_angle > 360):
        camera_y_angle = 0
    
      
    
pygame.display.quit()


Vertex position Count:  530 (5.929688, 4.125, 0.0)
Vertex normal Count:  530 (-0.966742, -0.255752, 0.0)
Vertex texture coordinate Count:  800 (2.0, 2.0, 0.0)
Face count:  992.0
texture exists
